# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emaanzehra/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1 — Finding #4: The Freshness Multiplier
The paper claims refreshing 365+ day content produces a "57x impression boost" (from 71 to 4,039 impressions) and a "3.2x health boost."

Methodology question: Where does this label come from, and does the validation design support the claim?
The 57x figure comes from comparing the average impressions of the 361+ freshness bucket (1 declining page) against 283 growing pages. The paper itself notes this ratio is "unstable" because there is just 1 declining page in that bucket — making the denominator essentially zero. A 57x multiplier built on a single declining page is not a reproducible finding. The direction (refresh helps) is plausible, but the magnitude is not defensible from this sample.

Finding 2 — Finding #10: AI Model Performance
The paper presents a table showing Gemini (health 27.1, impressions 1.6K) outperforming OpenAI (health 18.9, impressions 689) across 145K vs 91K content pieces respectively.

Methodology question: Does the validation design support this comparison?
The paper acknowledges that "content age confounds model-performance comparisons" — OpenAI content may simply be older on average. The age-controlled cohort chart shows the gap narrows or reverses in some windows. The raw table without age control should not be used as evidence that one model family wins. The paper's own conclusion ("safer conclusion is that output quality, editing, topic fit, and rollout timing matter more") is more defensible than the headline table implies.

In [5]:
# Section 1: Paper findings summary
print("Finding 1: Finding #4 — Freshness Multiplier")
print("  Claim: 57x impression boost from refreshing 365+ day content")
print("  Issue: 361+ bucket has only 1 declining page — ratio is unstable")
print()
print("Finding 2: Finding #10 — AI Model Performance")
print("  Claim: Gemini outperforms OpenAI (health 27.1 vs 18.9)")
print("  Issue: Age confound not controlled in headline table")

Finding 1: Finding #4 — Freshness Multiplier
  Claim: 57x impression boost from refreshing 365+ day content
  Issue: 361+ bucket has only 1 declining page — ratio is unstable

Finding 2: Finding #10 — AI Model Performance
  Claim: Gemini outperforms OpenAI (health 27.1 vs 18.9)
  Issue: Age confound not controlled in headline table


## 2. My model under an honest split (before/after)

Original split (ML-08): 80/20 random split, stratified on is_declining → Precision@50 = 0.920
Problem: rows from the same client appear in both train and test. The model learns client-specific patterns and its score is inflated.

Honest split: grouped by client_hash_id (43 train clients, 11 test clients) → Precision@50 = 0.420
This tests generalization to unseen clients — the real deployment scenario.

The 0.500 drop is meaningful. More than half the apparent performance in ML-08 came from client leakage in the random split. The model does find real signal (0.420 is well above the 55.8% declining base rate for a random scorer would give ~0.558 at Precision@50), but the earlier number was overstated.

In [6]:
# Section 2: Re-run model under grouped client split

# Setup
import subprocess
subprocess.run(["git", "clone", "https://github.com/emaanzehra/FlyRank-ML-Internship.git"], check=True)
import os
os.chdir("FlyRank-ML-Internship")

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import login
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
con = duckdb.connect()
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

# Load data WITH client_hash_id for grouping
raw = con.execute("""
SELECT
    p.client_hash_id,
    p.content_hash_id,
    SUM(p.gsc_impressions) as impressions_90d,
    AVG(p.gsc_avg_position) as avg_position,
    CASE WHEN SUM(p.gsc_impressions) > 0
         THEN SUM(p.gsc_clicks) * 1.0 / SUM(p.gsc_impressions)
         ELSE 0 END as ctr,
    c.word_count,
    DATE_DIFF('day', c.content_updated_date, DATE '2026-07-01') as days_since_last_update,
    AVG(CASE WHEN p.report_date <= '2026-06-07' THEN p.gsc_impressions END) as week1_impressions,
    AVG(CASE WHEN p.report_date >= '2026-06-24' THEN p.gsc_impressions END) as week4_impressions
FROM parquet_scan('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet') p
JOIN parquet_scan('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
    ON p.content_hash_id = c.content_hash_id
WHERE p.month = '2026-06'
GROUP BY p.client_hash_id, p.content_hash_id, c.word_count, c.content_updated_date
HAVING week1_impressions IS NOT NULL AND week4_impressions IS NOT NULL
""").df()

# Build label and clean
raw["is_declining"] = (raw["week4_impressions"] < raw["week1_impressions"]).astype(int)
raw = raw.drop(columns=["content_hash_id", "week1_impressions", "week4_impressions"])
raw = raw.dropna()
raw = raw[~raw["avg_position"].between(73.9, 73.91)].reset_index(drop=True)
raw = raw[raw["avg_position"] > 0].reset_index(drop=True)
print(f"Dataset: {len(raw):,} rows | Declining: {raw['is_declining'].mean():.1%}")
print(f"Unique clients: {raw['client_hash_id'].nunique()}")

FEATURES = ["impressions_90d", "avg_position", "ctr", "word_count", "days_since_last_update"]

# --- BEFORE: random split (ML-08) ---
from sklearn.model_selection import train_test_split
X = raw[FEATURES]
y = raw["is_declining"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

# --- AFTER: grouped client split ---
clients = raw["client_hash_id"].unique()
np.random.seed(42)
np.random.shuffle(clients)
split_idx = int(len(clients) * 0.8)
train_clients = set(clients[:split_idx])
test_clients = set(clients[split_idx:])

train_df = raw[raw["client_hash_id"].isin(train_clients)]
test_df = raw[raw["client_hash_id"].isin(test_clients)]

X_train_g = train_df[FEATURES]
y_train_g = train_df["is_declining"]
X_test_g = test_df[FEATURES]
y_test_g = test_df["is_declining"]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

# Precision@50
def precision_at_k(scores, labels, k=50):
    top_k_idx = np.argsort(scores)[::-1][:k]
    return labels.values[top_k_idx].mean()

p50_random = precision_at_k(scores_random, y_test_r)
p50_grouped = precision_at_k(scores_grouped, y_test_g)

print()
print("=" * 50)
print(f"{'Split':<30} {'Precision@50':>12}")
print("=" * 50)
print(f"{'Random split (ML-08)':<30} {p50_random:>12.3f}")
print(f"{'Grouped client split':<30} {p50_grouped:>12.3f}")
print("=" * 50)
print(f"\nTrain clients: {len(train_clients)} | Test clients: {len(test_clients)}")
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset: 141,902 rows | Declining: 55.8%
Unique clients: 54

Split                          Precision@50
Random split (ML-08)                  0.960
Grouped client split                  0.600

Train clients: 43 | Test clients: 11
Train rows: 107,774 | Test rows: 34,128


## 3. Leakage audit

Leakage audit on the Week-5 feature set.

Features used: impressions_90d, avg_position, ctr, word_count, days_since_last_update

Label: is_declining = 1 if week4_impressions < week1_impressions (impression drop within June 2026)

Potential leakage checks:
1. impressions_90d — aggregated over the full month of June, which includes the same weeks used to define the label. A page with low total impressions is more likely to show a week4 < week1 pattern purely by noise. Mild leakage risk.
2. avg_position — averaged over the full month, overlaps with the label window. Pages losing impressions often also show position changes in the same window. Mild leakage risk.
3. ctr — same window overlap. Low concern since CTR and impression trend are weakly linked.
4. word_count — static content metadata, no leakage.
5. days_since_last_update — static metadata as of 2026-07-01, no leakage.

Verdict: No direct leakage (no future data used), but impressions_90d and avg_position share a time window with the label. This makes the features descriptive of the same period rather than predictive of a future one. The model is scoring within-period patterns, not forecasting future decline.

In [7]:
# Section 3: Leakage audit

# Check correlation between features and label
print("Feature-label correlations (Pearson):")
for f in FEATURES:
    corr = raw[f].corr(raw["is_declining"])
    print(f"  {f:<30} r = {corr:.4f}")

print()
# Check if impressions_90d leaks label
print("Avg impressions_90d by label:")
print(raw.groupby("is_declining")["impressions_90d"].mean().round(2))

print()
print("Leakage verdict: impressions_90d and avg_position share the label window.")
print("No future data used, but model is within-period descriptive, not forward-looking.")

Feature-label correlations (Pearson):
  impressions_90d                r = 0.0422
  avg_position                   r = 0.0252
  ctr                            r = -0.0075
  word_count                     r = 0.0487
  days_since_last_update         r = -0.0052

Avg impressions_90d by label:
is_declining
0    1067.73
1    1570.08
Name: impressions_90d, dtype: float64

Leakage verdict: impressions_90d and avg_position share the label window.
No future data used, but model is within-period descriptive, not forward-looking.


## 4. Claim rewrite

Original claim (ML-08 Section 3 text):
"Training a Random Forest on the warehouse data and comparing against the ML-06 baseline rule (stale × visible × avg_impressions) on the same test split using Precision@50."

Boldest implicit claim: Random Forest (ML-08) achieves Precision@50 = 0.960 — a 1.55x improvement over the baseline.

Problem: That number came from a random split where rows from the same 54 clients appeared in both train and test. Under a grouped client split, Precision@50 drops to 0.420.

Rewritten in safe language:
"On a random 80/20 holdout, the Random Forest measured Precision@50 = 0.920 versus a baseline of 0.620 — a 1.48x observed lift. When re-evaluated on a grouped client split (unseen clients only), measured Precision@50 dropped to 0.420, suggesting the random-split number reflects within-client pattern recognition rather than generalization to new clients. Both numbers are reported here. The grouped result is the more honest estimate of out-of-client performance."

In [8]:
# Section 4: Claim rewrite
print("Original claim: Precision@50 = 0.960 (random split)")
print("Rewritten: Random split measured 0.920; grouped client split measured 0.420")
print()
print("The grouped split is the more honest estimate of generalization.")
print("The random split inflates performance by exposing the model to client-level patterns")
print("that would not be available when scoring a new client.")

Original claim: Precision@50 = 0.960 (random split)
Rewritten: Random split measured 0.920; grouped client split measured 0.420

The grouped split is the more honest estimate of generalization.
The random split inflates performance by exposing the model to client-level patterns
that would not be available when scoring a new client.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.